In [1]:
import pandas as pd
import srsly
from html import unescape

In [ ]:
import pandas as pd
import gzip

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

df = getDF('./meta_Beauty.json.gz')

In [ ]:
main = srsly.read_jsonl("./amazon2014/reviews_Beauty_5.json")

In [4]:
main_data = pd.read_csv("./formatted_data.csv")

In [5]:
main_data = pd.DataFrame(main)

In [6]:
result = main_data.merge(df, on = 'asin',how = "left")

In [7]:
result.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,description,title,imUrl,salesRank,categories,price,related,brand
0,A1YJEY40YUW4SE,7806397051,Andrea,"[3, 4]",Very oily and creamy. Not at all what I expect...,1.0,Don't waste your money,1391040000,"01 30, 2014",An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA
1,A60XNB876KYML,7806397051,Jessica H.,"[1, 1]",This palette was a decent price and I was look...,3.0,OK Palette!,1397779200,"04 18, 2014",An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA
2,A3G6XNM240RMWA,7806397051,Karen,"[0, 1]",The texture of this concealer pallet is fantas...,4.0,great quality,1378425600,"09 6, 2013",An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA
3,A1PQFP6SAJ6D80,7806397051,Norah,"[2, 2]",I really can't tell what exactly this thing is...,2.0,Do not work on my face,1386460800,"12 8, 2013",An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA
4,A38FVHZTNQ271F,7806397051,Nova Amor,"[0, 0]","It was a little smaller than I expected, but t...",3.0,It's okay.,1382140800,"10 19, 2013",An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,{'Beauty': 10486},"[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA


In [16]:
data_sorted = result.sort_values(by=['reviewerID', 'unixReviewTime'])

grouped_data = data_sorted.groupby('reviewerID').apply(
    lambda x: {
        'reviewerID': x['reviewerID'].tolist(),
        'title': x['title'].tolist(),
        'overall': x['overall'].tolist(),
        'unixReviewTime': x['unixReviewTime'].tolist(),
        'asin': x['asin'].tolist()
    }
).reset_index(name='data')

grouped_data.head(10)

/tmp/ipykernel_236680/3053444316.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_data = data_sorted.groupby('reviewerID').apply(


,reviewerID,data
0,A00414041RD0BXM6WK0GX,"{'reviewerID': ['A00414041RD0BXM6WK0GX', 'A004..."
1,A00473363TJ8YSZ3YAGG9,"{'reviewerID': ['A00473363TJ8YSZ3YAGG9', 'A004..."
2,A00700212KB3K0MVESPIY,"{'reviewerID': ['A00700212KB3K0MVESPIY', 'A007..."
3,A0078719IR14X3NNUG0F,"{'reviewerID': ['A0078719IR14X3NNUG0F', 'A0078..."
4,A01198201H0E3GHV2Z17I,"{'reviewerID': ['A01198201H0E3GHV2Z17I', 'A011..."
5,A02155413BVL8D0G7X6DN,"{'reviewerID': ['A02155413BVL8D0G7X6DN', 'A021..."
6,A02849582PREZYNEI31CV,"{'reviewerID': ['A02849582PREZYNEI31CV', 'A028..."
7,A029527620Q3SK5XW16RR,"{'reviewerID': ['A029527620Q3SK5XW16RR', 'A029..."
8,A03236882LUP4ARMDDMXU,"{'reviewerID': ['A03236882LUP4ARMDDMXU', 'A032..."
9,A03364251DGXSGA9PSR99,"{'reviewerID': ['A03364251DGXSGA9PSR99', 'A033..."


In [17]:
# ID part
grouped_data['prompt'] = grouped_data['data'].apply(
    lambda x: f"On the Amazon platform, user_{x['reviewerID'][0]} " + ",".join( 
        [f"rated the item titled {asin} with a rating of {overall}"
     for asin, overall in zip( x['asin'], x['overall'])]) 
     )

grouped_data = grouped_data.drop(columns=['data'])
grouped_data.head()

,reviewerID,prompt
0,A00414041RD0BXM6WK0GX,"On the Amazon platform, user_A00414041RD0BXM6W..."
1,A00473363TJ8YSZ3YAGG9,"On the Amazon platform, user_A00473363TJ8YSZ3Y..."
2,A00700212KB3K0MVESPIY,"On the Amazon platform, user_A00700212KB3K0MVE..."
3,A0078719IR14X3NNUG0F,"On the Amazon platform, user_A0078719IR14X3NNU..."
4,A01198201H0E3GHV2Z17I,"On the Amazon platform, user_A01198201H0E3GHV2..."


In [18]:
from sklearn.preprocessing import LabelEncoder
import re

le = LabelEncoder()
grouped_data['encoded_reviewerID'] = le.fit_transform(grouped_data['reviewerID'])

id_mapping = dict(zip(grouped_data['reviewerID'], grouped_data['encoded_reviewerID']))

def replace_reviewerID(prompt, mapping):
    # 正则表达式匹配 user_ 后面的 reviewerID
    match = re.search(r"user_(\w+)", prompt)
    if match:
        original_id = match.group(1)  # 提取原始 reviewerID
        encoded_id = mapping.get(original_id, original_id)  # 获取映射后的 ID
        return re.sub(r"user_\w+", f"user_{encoded_id}", prompt)  # 替换为编码后的 ID
    return prompt

grouped_data['prompt'] = grouped_data['prompt'].apply(lambda x: replace_reviewerID(x, id_mapping))

grouped_data = grouped_data.drop(columns=['reviewerID', 'encoded_reviewerID'])

grouped_data.head(20)

,prompt
0,"On the Amazon platform, user_0 rated the item ..."
1,"On the Amazon platform, user_1 rated the item ..."
2,"On the Amazon platform, user_2 rated the item ..."
3,"On the Amazon platform, user_3 rated the item ..."
4,"On the Amazon platform, user_4 rated the item ..."
5,"On the Amazon platform, user_5 rated the item ..."
6,"On the Amazon platform, user_6 rated the item ..."
7,"On the Amazon platform, user_7 rated the item ..."
8,"On the Amazon platform, user_8 rated the item ..."
9,"On the Amazon platform, user_9 rated the item ..."


In [19]:
def clean_nan_items(prompt):
    cleaned_prompt = re.sub(r'rated the item titled nan with a rating of \d+\.\d+,?', '', prompt)
    return cleaned_prompt.strip(', ')  # 移除开头或结尾多余的逗号和空格
grouped_data['prompt'] = grouped_data['prompt'].apply(clean_nan_items)


In [21]:
grouped_data['prompt'] = grouped_data['prompt'].apply(unescape)
grouped_data[['prompt']].to_csv('collaborative_prompts.csv', index=False)
print("prompt 列已成功保存为 collaborative_prompts.csv 文件！")

prompt 列已成功保存为 collaborative_prompts.csv 文件！


In [23]:
grouped_data.to_csv('./collaborative.csv')

In [ ]:
grouped_data['prompt'] = grouped_data['data'].apply(
    lambda x: f"On the Amazon platform, user_{x['reviewerID'][0]} " + ",".join( 
        [f"rated the item titled {title} with a rating of {overall}"
     for title, overall in zip( x['title'], x['overall'])]) 
     )

grouped_data = grouped_data.drop(columns=['data'])
grouped_data.head()

,reviewerID,prompt
0,A00414041RD0BXM6WK0GX,"On the Amazon platform, user_A00414041RD0BXM6W..."
1,A00473363TJ8YSZ3YAGG9,"On the Amazon platform, user_A00473363TJ8YSZ3Y..."
2,A00700212KB3K0MVESPIY,"On the Amazon platform, user_A00700212KB3K0MVE..."
3,A0078719IR14X3NNUG0F,"On the Amazon platform, user_A0078719IR14X3NNU..."
4,A01198201H0E3GHV2Z17I,"On the Amazon platform, user_A01198201H0E3GHV2..."


In [10]:
from sklearn.preprocessing import LabelEncoder
import re

le = LabelEncoder()
grouped_data['encoded_reviewerID'] = le.fit_transform(grouped_data['reviewerID'])

id_mapping = dict(zip(grouped_data['reviewerID'], grouped_data['encoded_reviewerID']))

def replace_reviewerID(prompt, mapping):
    # 正则表达式匹配 user_ 后面的 reviewerID
    match = re.search(r"user_(\w+)", prompt)
    if match:
        original_id = match.group(1)  # 提取原始 reviewerID
        encoded_id = mapping.get(original_id, original_id)  # 获取映射后的 ID
        return re.sub(r"user_\w+", f"user_{encoded_id}", prompt)  # 替换为编码后的 ID
    return prompt

grouped_data['prompt'] = grouped_data['prompt'].apply(lambda x: replace_reviewerID(x, id_mapping))

grouped_data = grouped_data.drop(columns=['reviewerID', 'encoded_reviewerID'])

grouped_data.head(20)

,prompt
0,"On the Amazon platform, user_0 rated the item ..."
1,"On the Amazon platform, user_1 rated the item ..."
2,"On the Amazon platform, user_2 rated the item ..."
3,"On the Amazon platform, user_3 rated the item ..."
4,"On the Amazon platform, user_4 rated the item ..."
5,"On the Amazon platform, user_5 rated the item ..."
6,"On the Amazon platform, user_6 rated the item ..."
7,"On the Amazon platform, user_7 rated the item ..."
8,"On the Amazon platform, user_8 rated the item ..."
9,"On the Amazon platform, user_9 rated the item ..."


In [11]:
def clean_nan_items(prompt):
    cleaned_prompt = re.sub(r'rated the item titled nan with a rating of \d+\.\d+,?', '', prompt)
    return cleaned_prompt.strip(', ')  # 移除开头或结尾多余的逗号和空格
grouped_data['prompt'] = grouped_data['prompt'].apply(clean_nan_items)


In [12]:
grouped_data['prompt'] = grouped_data['prompt'].apply(unescape)
grouped_data[['prompt']].to_csv('grouped_prompts.csv', index=False)
print("prompt 列已成功保存为 grouped_prompts.csv 文件！")

prompt 列已成功保存为 grouped_prompts.csv 文件！


In [ ]:
grouped_data.to_csv('./semantic.csv')

In [12]:
import pandas as pd
data = pd.read_csv("./grouped_prompts.csv")
p=data.iloc[0]["prompt"]

In [13]:
data.iloc[0]["prompt"]

'On the Amazon platform, user_0 rated the item titled 63cm Long Zipper Beige+pink Wavy Cosplay Hair Wig Rw157 with a rating of 3.0,rated the item titled MapofBeauty Long Wave Curly Hair Wig Full Wig for Women Long (Black) with a rating of 2.0,rated the item titled MapofBeauty Cosplay Costume Long Curly Hair Wig Ladies Synthetic Wigs (White) with a rating of 1.0,rated the item titled 32" 80cm Long Hair Heat Resistant Spiral Curly Cosplay Wig (Red Dark) with a rating of 3.0,rated the item titled MapofBeauty 28" 70cm Long Curly Hair Ends Costume Cosplay Wig (Brown) with a rating of 5.0,rated the item titled MapofBeauty Beautiful Women\'s Flat Bang Long Wave Curly Wig (Black) with a rating of 5.0'

In [27]:
user_behavior_prompt = """
You are an expert in user behavior analysis, specializing in identifying long-term and short-term interests from chronological user interaction data. 

Your goal is to analyze the given user interaction sequence and infer both long-term and short-term interests according to the rules below:

Analysis Rules
1. **Long-term interest**:
   - Reflects consistent engagement across the entire interaction history.
   - Derived by analyzing patterns such as recurring themes or categories in interactions.
    Examples: If a user frequently engages with movies like Terminator or Dune, the inferred long-term interest might be categories like “Science Fiction” or “War”
   - **MUST infer exactly one (1) category as long-term interest.**

2. **Short-term interest**:
   - Reflects concentrated but temporary engagement in a specific timeframe.
   - Represents the user's current or situational preferences, which might be influenced by trends, events, or recent needs.
   - Derived by analyzing recent interactions or clusters of interest within the most recent data points.
   Examples: If a user recently interacted with content related to travel or holiday deals, the inferred short-term interest might be “Travel Planning” or “Vacation”.
   - **Output multiple categories (1 or more) based on the detected short-term interests.**

Guidelines for Analysis
	•	Use chronological product interaction data to segment the interests into long-term and short-term categories.
	•	Ensure that the long-term and short-term interests are meaningful, concise, and directly relevant to the user interaction data.

**Output Format**:
```The response must follow the structure below, formatted as CSV:

Category,Type
[long_term_interest1, long_term_interest2,...],Long-term

[short_term_interest_1, short_term_interest_2,...],Short-term

Note: 
 -Both long-term and short-term interests MUST be filled. The response must be valid CSV format.
 -Replace Category with the inferred labels (e.g., “Science Fiction”, “Travel Planning”).
 -Ensure the long-term interest appears only once under the type “Long-term”
 -Be concise and avoid redundancy in category naming.
"""

p = user_behavior_prompt+'\n\n'+p